# MSMARCO-XI Full-Coverage Production RAG Indexer — Nepali (नेपाली)

This notebook is the full-dataset architecture for the final multilingual RAG system configured for **Nepali (`ne` / `nep`)**.

**Coverage:** every source record is indexed. `is_selected` is not used to delete records.

**Design:**
`MSMARCO-XI (Nepali: train/neptrain.parquet) → record-level dense index → candidate records → multi-strategy passage chunking → grounded answer`

Each language is a separate resumable Kaggle job. This prevents one long all-language run from timing out.

> **Important Kaggle Setting:** Ensure **Internet ON** in Notebook Settings (right sidebar: *Notebook Options -> Internet -> ON*), or attach MSMARCO-XI as a Kaggle Input dataset.


## 1. Safe Kaggle environment

In [ ]:
import sys, importlib.util, os
import numpy as np
import pandas as pd
import scipy
import torch

print("Python:", sys.version.split()[0])
print("NumPy :", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy :", scipy.__version__)
print("Torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU REQUIRED. Kaggle Settings → Accelerator → GPU, then restart.")

print("GPU   :", torch.cuda.get_device_name(0))
torch.set_float32_matmul_precision("high")

# Try importing faiss, or try installing faiss-cpu if internet is available
FAISS_AVAILABLE = False
try:
    import faiss
    FAISS_AVAILABLE = True
    print("FAISS :", getattr(faiss, "__version__", "available"))
except ImportError:
    try:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
        import faiss
        FAISS_AVAILABLE = True
        print("FAISS :", getattr(faiss, "__version__", "installed & available"))
    except Exception:
        FAISS_AVAILABLE = False
        print("FAISS : not installed — using high-speed GPU Torch vector-store fallback.")

## 2. Hugging Face token

In [ ]:
import os
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print("HF token available:", bool(HF_TOKEN))

## 3. Full-coverage configuration

In [ ]:
from pathlib import Path

LANGUAGE_NAMES = {
    "as":"Assamese","bn":"Bengali","gu":"Gujarati","hi":"Hindi",
    "kn":"Kannada","ml":"Malayalam","mr":"Marathi","ne":"Nepali",
    "or":"Odia","pa":"Punjabi","sa":"Sanskrit","ta":"Tamil",
    "te":"Telugu","ur":"Urdu",
}
LANGUAGE_PREFIXES = {
    "as":"asm","bn":"ben","gu":"guj","hi":"hin","kn":"kan",
    "ml":"mal","mr":"mar","ne":"nep","or":"ori","pa":"pan",
    "sa":"san","ta":"tam","te":"tel","ur":"urd",
}

LANGUAGE = "ne"          # change for each Kaggle job
SPLIT = "train"
PASSAGE_MODE = "translated"

FULL_RUN = True           # ALL rows for this language
RECORD_TEXT_MAX_CHARS = 7000

# Large-scale index: approximately one vector per source record.
FAISS_NLIST = 4096
FAISS_PQ_M = 48
FAISS_PQ_BITS = 8
FAISS_TRAIN_RECORDS = 200_000

# Candidate-stage chunking options
FIXED_SIZE = 500
FIXED_OVERLAP = 80
SENTENCES_PER_CHUNK = 3
SEMANTIC_THRESHOLD = 0.58

BATCH_ROWS = 1024
EMBED_BATCH_SIZE = 512
ROWS_PER_RECORD_SHARD = 100_000

MODEL_NAME = "intfloat/multilingual-e5-small"

ROOT = Path("/kaggle/working/msmarco_xi_full")
LANG_ROOT = ROOT / LANGUAGE
RECORD_ROOT = LANG_ROOT / "records"
RECORD_ROOT.mkdir(parents=True, exist_ok=True)

print("Language:", LANGUAGE_NAMES[LANGUAGE])
print("FULL RUN:", FULL_RUN)

## System card — values to use when answering implementation questions

### Embedding
- Model: `intfloat/multilingual-e5-small`
- Vector dimension: **384**
- Distance/similarity: **inner product** on normalized embeddings (cosine-equivalent)
- Indexing input prefix: `passage: `
- Query input prefix: `query: `
- The record-level embedding text is built from:
  1. source query
  2. source answer
  3. selected translated passages when available
  4. otherwise the first available translated passages
- The full original passages are preserved separately in compressed Parquet record shards.

### Current full-index chunking status
The **full 778k-record index is record-level, not chunk-level**: approximately one vector per source record.

The notebook also contains four **candidate-stage** chunking strategies for the RAG retrieval stage:
1. Sentence-aware
2. Fixed-size with overlap
3. Semantic
4. Metadata-aware

These are applied to passages from the retrieved candidate records before final reranking/LLM context selection. They are **not all materialized as four giant vector databases**.

### FAISS
- Type: IVF-PQ
- `nlist = 4096`
- Product quantization: `48 x 8-bit`
- One vector per source record in this full-coverage index.

### Current language in this notebook
**Nepali (`ne` / `nep`)**

## 4. Open the exact remote Parquet file

In [ ]:
import os, glob
from pathlib import Path
import pyarrow.parquet as pq

repo_file = f"{SPLIT}/{LANGUAGE_PREFIXES[LANGUAGE]}train.parquet"
if SPLIT != "train":
    repo_file = f"{SPLIT}/{LANGUAGE_PREFIXES[LANGUAGE]}val.parquet"

parquet_file = None
SOURCE_ROWS = 0

# ── 1. Check local Kaggle input datasets first (/kaggle/input/...) ───────────
candidate_local_files = [
    Path(f"/kaggle/input/msmarco-xi/{repo_file}"),
    Path(f"/kaggle/input/msmarco-xi/{LANGUAGE_PREFIXES[LANGUAGE]}train.parquet"),
    Path(f"/kaggle/input/{LANGUAGE_PREFIXES[LANGUAGE]}train.parquet"),
    Path(f"./data_cache/{repo_file}"),
]
for p in Path("/kaggle/input").rglob(f"*{LANGUAGE_PREFIXES[LANGUAGE]}*parquet*"):
    candidate_local_files.append(p)

for local_path in candidate_local_files:
    if local_path.exists() and local_path.is_file():
        try:
            parquet_file = pq.ParquetFile(str(local_path))
            SOURCE_ROWS = int(parquet_file.metadata.num_rows)
            print(f"Loaded from local Kaggle dataset: {local_path} ({SOURCE_ROWS:,} rows)")
            break
        except Exception:
            pass

# ── 2. Stream from Hugging Face Hub (No redundant repo_info call) ─────────────
if parquet_file is None:
    from huggingface_hub import hf_hub_url, hf_hub_download
    import fsspec
    
    url = hf_hub_url(
        repo_id="ai4bharat/MSMARCO-XI",
        filename=repo_file,
        repo_type="dataset",
        revision="main",
    )
    
    headers = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}
    
    # Try 1: Remote stream via fsspec
    try:
        print(f"Connecting to remote parquet stream for {LANGUAGE_NAMES[LANGUAGE]}: {repo_file}...")
        fs = fsspec.filesystem("https", headers=headers)
        remote_handle = fs.open(url, "rb", block_size=16 * 1024 * 1024, cache_type="readahead")
        parquet_file = pq.ParquetFile(remote_handle)
        SOURCE_ROWS = int(parquet_file.metadata.num_rows)
        print(f"Streaming from Hugging Face: {repo_file} ({SOURCE_ROWS:,} rows)")
    except Exception as e1:
        print(f"Notice on fsspec streaming ({e1}). Trying hf_hub_download fallback...")
        try:
            dl_path = hf_hub_download(
                repo_id="ai4bharat/MSMARCO-XI",
                filename=repo_file,
                repo_type="dataset",
                token=HF_TOKEN,
            )
            parquet_file = pq.ParquetFile(dl_path)
            SOURCE_ROWS = int(parquet_file.metadata.num_rows)
            print(f"Downloaded via hf_hub_download: {dl_path} ({SOURCE_ROWS:,} rows)")
        except Exception as e2:
            raise RuntimeError(
                f"Failed to access '{repo_file}' from Hugging Face Hub.\n"
                f"ERROR: {e2}\n\n"
                f"--> HOW TO FIX IN KAGGLE:\n"
                f"1. In the right sidebar of your Kaggle notebook, open 'Notebook Options'.\n"
                f"2. Toggle 'Internet' to ON (requires phone-verified Kaggle account).\n"
                f"3. Re-run this cell."
            )

print("File:", repo_file)
print("Source rows:", f"{SOURCE_ROWS:,}")

## 5. GPU embedding model

In [ ]:
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,
).to(device)
model.eval()

@torch.inference_mode()
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).expand(hidden.size()).float()
    return (hidden * m).sum(1) / torch.clamp(m.sum(1), min=1e-9)

@torch.inference_mode()
def encode_texts(texts, prefix="passage: "):
    if not texts:
        return np.empty((0, model.config.hidden_size), dtype="float32")
    all_vecs = []
    for i in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = [prefix + str(x) for x in texts[i:i+EMBED_BATCH_SIZE]]
        tokens = tokenizer(
            batch, padding=True, truncation=True, max_length=384, return_tensors="pt"
        )
        tokens = {k: v.to(device, non_blocking=True) for k, v in tokens.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(**tokens)
            emb = mean_pool(out.last_hidden_state, tokens["attention_mask"])
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        all_vecs.append(emb.float().cpu().numpy().astype("float32"))
    return np.vstack(all_vecs)

print("Embedding test:", encode_texts(["test"]).shape)

## 6. Record representation: one vector per source record, all passages preserved

In [ ]:
def build_record_text(record):
    passages = record.get("passages") or {}
    if PASSAGE_MODE == "translated":
        texts = passages.get("Translated_passages") or []
    else:
        texts = passages.get("English_passages") or []

    selected = passages.get("is_selected") or []
    selected_texts = [
        str(texts[i]).strip()
        for i, flag in enumerate(selected)
        if flag == 1 and i < len(texts) and str(texts[i]).strip()
    ]
    if not selected_texts:
        selected_texts = [str(x).strip() for x in texts[:2] if str(x).strip()]

    pieces = [
        str(record.get("query", "")).strip(),
        str(record.get("Answer", "")).strip(),
        *selected_texts,
    ]
    return "\n".join(x for x in pieces if x)[:RECORD_TEXT_MAX_CHARS]

def normalize_record(record, local_id):
    passages = record.get("passages") or {}
    return {
        "local_id": int(local_id),
        "query_id": int(record.get("query_id", 0)),
        "query": str(record.get("query", "")),
        "answer": str(record.get("Answer", "")),
        "query_type": str(record.get("query_type", "")),
        "source_lang": str(record.get("source_lang", "")),
        "target_lang": str(record.get("target_lang", "")),
        "english_passages": [str(x) for x in passages.get("English_passages") or []],
        "translated_passages": [str(x) for x in passages.get("Translated_passages") or []],
        "is_selected": [int(x) for x in passages.get("is_selected") or []],
    }

## 7. Train IVF-PQ from a sample

In [ ]:
import time, json
DIM = int(model.config.hidden_size)

INDEX_PATH = LANG_ROOT / "faiss_ivfpq.index"
VECTOR_PATH = LANG_ROOT / "record_vectors.float16.bin"
CHECKPOINT_PATH = LANG_ROOT / "checkpoint.json"
CONFIG_PATH = LANG_ROOT / "config.json"

if FAISS_AVAILABLE:
    train_texts = []
    for batch in parquet_file.iter_batches(
        batch_size=2048,
        columns=["query","Answer","passages"],
    ):
        for record in batch.to_pylist():
            text = build_record_text(record)
            if text:
                train_texts.append(text)
            if len(train_texts) >= FAISS_TRAIN_RECORDS:
                break
        if len(train_texts) >= FAISS_TRAIN_RECORDS:
            break

    train_vectors = encode_texts(train_texts)
    quantizer = faiss.IndexFlatIP(DIM)
    index = faiss.IndexIVFPQ(
        quantizer, DIM, FAISS_NLIST, FAISS_PQ_M, FAISS_PQ_BITS,
        faiss.METRIC_INNER_PRODUCT
    )

    t0 = time.perf_counter()
    index.train(train_vectors)
    training_seconds = time.perf_counter() - t0

    ENGINE = "FAISS_IVFPQ"
    print("IVF-PQ trained.")
    print("Training records:", len(train_texts))
    print("Training seconds:", round(training_seconds, 1))
    print("nlist:", FAISS_NLIST, "PQ:", f"{FAISS_PQ_M}x{FAISS_PQ_BITS}")

else:
    # Exact normalized vector fallback. 778,638 × 384 × float16 ≈ 570 MiB.
    vector_memmap = np.memmap(
        VECTOR_PATH,
        mode="w+",
        dtype=np.float16,
        shape=(SOURCE_ROWS, DIM),
    )
    vector_memmap.flush()
    index = None
    training_seconds = 0.0
    ENGINE = "TORCH_GPU_EXACT_FLOAT16"
    print("Using Torch GPU fallback.")
    print("Vector store:", VECTOR_PATH)
    print("Estimated size:",
          round(SOURCE_ROWS * DIM * 2 / (1024**3), 2), "GiB")

## 8. Compressed source-record store

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

schema = pa.schema([
    ("local_id", pa.int64()),
    ("query_id", pa.int64()),
    ("query", pa.string()),
    ("answer", pa.string()),
    ("query_type", pa.string()),
    ("source_lang", pa.string()),
    ("target_lang", pa.string()),
    ("english_passages", pa.list_(pa.string())),
    ("translated_passages", pa.list_(pa.string())),
    ("is_selected", pa.list_(pa.int8())),
])

def write_shard(rows, shard_id):
    if not rows:
        return
    path = RECORD_ROOT / f"records_{shard_id:05d}.parquet"
    table = pa.Table.from_pydict(
        {k: [r[k] for r in rows] for k in schema.names},
        schema=schema,
    )
    pq.write_table(
        table, path,
        compression="zstd",
        compression_level=7,
        use_dictionary=True,
    )

## 9. Full indexing loop with checkpoint/resume

In [ ]:
import json, time
from tqdm.auto import tqdm

next_id = 0
if CHECKPOINT_PATH.exists():
    cp = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
    next_id = int(cp.get("next_id", 0))
    if ENGINE == "FAISS_IVFPQ" and INDEX_PATH.exists():
        index = faiss.read_index(str(INDEX_PATH))
    print("Resuming at local_id:", next_id)

pending_texts, pending_ids, pending_records = [], [], []
shard_rows = []
shard_id = next_id // ROWS_PER_RECORD_SHARD
started = time.perf_counter()

def flush():
    global pending_texts, pending_ids, pending_records, shard_rows, shard_id, next_id

    if not pending_texts:
        return

    vecs = encode_texts(pending_texts, prefix="passage: ")
    ids = np.asarray(pending_ids, dtype=np.int64)

    if ENGINE == "FAISS_IVFPQ":
        index.add_with_ids(vecs, ids)
    else:
        vector_memmap[ids[0]:ids[0] + len(ids)] = vecs.astype(np.float16)

    shard_rows.extend(pending_records)

    while len(shard_rows) >= ROWS_PER_RECORD_SHARD:
        write_shard(shard_rows[:ROWS_PER_RECORD_SHARD], shard_id)
        del shard_rows[:ROWS_PER_RECORD_SHARD]
        shard_id += 1

    pending_texts.clear()
    pending_ids.clear()
    pending_records.clear()

    if ENGINE == "FAISS_IVFPQ":
        faiss.write_index(index, str(INDEX_PATH))
    else:
        vector_memmap.flush()

    CHECKPOINT_PATH.write_text(
        json.dumps(
            {
                "next_id": int(next_id),
                "vectors": int(next_id),
                "engine": ENGINE,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

for batch in tqdm(
    parquet_file.iter_batches(
        batch_size=BATCH_ROWS,
        columns=[
            "source_lang","target_lang","Answer","query_id",
            "query_type","passages","query"
        ],
    ),
    desc=f"FULL {LANGUAGE_NAMES[LANGUAGE]}",
):
    for record in batch.to_pylist():
        pending_texts.append(build_record_text(record))
        pending_ids.append(next_id)
        pending_records.append(normalize_record(record, next_id))
        next_id += 1

        if len(pending_texts) >= EMBED_BATCH_SIZE:
            flush()

flush()

if shard_rows:
    write_shard(shard_rows, shard_id)

if ENGINE == "FAISS_IVFPQ":
    faiss.write_index(index, str(INDEX_PATH))
else:
    vector_memmap.flush()

elapsed = time.perf_counter() - started

CONFIG_PATH.write_text(
    json.dumps(
        {
            "dataset": "ai4bharat/MSMARCO-XI",
            "language": LANGUAGE,
            "source_file": repo_file,
            "source_rows": int(SOURCE_ROWS),
            "records_indexed": int(next_id),
            "all_records": True,
            "all_passages_preserved": True,
            "index": ENGINE,
            "dimension": DIM,
            "nlist": FAISS_NLIST if ENGINE == "FAISS_IVFPQ" else None,
            "pq_m": FAISS_PQ_M if ENGINE == "FAISS_IVFPQ" else None,
            "pq_bits": FAISS_PQ_BITS if ENGINE == "FAISS_IVFPQ" else None,
            "embedding_model": MODEL_NAME,
            "elapsed_seconds": elapsed,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("FULL COMPLETE")
print("Source rows:", f"{SOURCE_ROWS:,}")
print("Vectors:", f"{next_id:,}")
print("Engine:", ENGINE)
print("Elapsed:", round(elapsed, 1), "sec")

## 10. Candidate-stage chunking

All records are preserved, but the system does not need four giant vector indexes.

After record retrieval, the backend can apply:
- fixed-size + overlap
- sentence-aware
- semantic
- metadata-aware

to the passages of the top candidate records.

In [ ]:
import re

def fixed_chunks(text, size=FIXED_SIZE, overlap=FIXED_OVERLAP):
    text = str(text).strip()
    out, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if text[start:end].strip():
            out.append(text[start:end].strip())
        if end >= len(text):
            break
        start += size - overlap
    return out

def sentence_chunks(text):
    sentences = [s.strip() for s in re.split(r"(?<=[.!?।॥])\s+", str(text)) if s.strip()]
    return [
        " ".join(sentences[i:i + SENTENCES_PER_CHUNK])
        for i in range(0, len(sentences), SENTENCES_PER_CHUNK)
    ]

def metadata_aware_chunks(text, query_type, language):
    return [
        f"[type={query_type} language={language}] {chunk}"
        for chunk in sentence_chunks(text)
    ]

def semantic_chunks(text):
    sentences = [s.strip() for s in re.split(r"(?<=[.!?।॥])\s+", str(text)) if s.strip()]
    if len(sentences) <= 1:
        return sentences
    vectors = encode_texts(sentences)
    out, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        if float(np.dot(vectors[i-1], vectors[i])) < SEMANTIC_THRESHOLD:
            out.append(" ".join(current))
            current = [sentences[i]]
        else:
            current.append(sentences[i])
    if current:
        out.append(" ".join(current))
    return out

## 11. Retrieval benchmark

This benchmark is retrieval-only. The final RAG dashboard must separately measure:
STT, query embedding, FAISS, record lookup, chunk/rerank, guardrails, LLM TTFT, LLM total, and E2E total.

Do not claim the retrieval number is the full 200 ms result.

In [ ]:
import time, numpy as np, torch

TORCH_BLOCK_ROWS = 32768

if ENGINE == "FAISS_IVFPQ":
    final_index = faiss.read_index(str(INDEX_PATH))

def retrieve(query, top_k=20):
    q = encode_texts([query], prefix="query: ")[0]

    if ENGINE == "FAISS_IVFPQ":
        scores, ids = final_index.search(q.reshape(1, -1), top_k)
        return [
            (int(i), float(s))
            for i, s in zip(ids[0], scores[0])
            if i >= 0
        ]

    q_tensor = torch.from_numpy(q).to(device, dtype=torch.float16)
    best_scores = torch.empty(0, device=device, dtype=torch.float16)
    best_ids = torch.empty(0, device=device, dtype=torch.long)

    for start in range(0, int(SOURCE_ROWS), TORCH_BLOCK_ROWS):
        end = min(start + TORCH_BLOCK_ROWS, int(SOURCE_ROWS))
        block = torch.from_numpy(np.asarray(vector_memmap[start:end])).to(
            device, dtype=torch.float16
        )
        scores = torch.matmul(block, q_tensor)
        vals, inds = torch.topk(scores, min(top_k, scores.shape[0]))
        inds = inds + start

        merged_scores = torch.cat([best_scores, vals])
        merged_ids = torch.cat([best_ids, inds])
        k = min(top_k, merged_scores.shape[0])
        best_scores, pos = torch.topk(merged_scores, k)
        best_ids = merged_ids[pos]

    return [
        (int(i), float(s))
        for i, s in zip(best_ids.cpu().tolist(), best_scores.cpu().tolist())
    ]

queries = []
for batch in parquet_file.iter_batches(batch_size=512, columns=["query"]):
    for record in batch.to_pylist():
        q = str(record.get("query", "")).strip()
        if q:
            queries.append(q)
        if len(queries) >= 100:
            break
    if len(queries) >= 100:
        break

times = []
for q in queries:
    t0 = time.perf_counter()
    retrieve(q)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    times.append((time.perf_counter() - t0) * 1000)

arr = np.asarray(times)
print("Queries:", len(arr))
print("Engine:", ENGINE)
print(f"P50: {np.percentile(arr,50):.2f} ms")
print(f"P70: {np.percentile(arr,70):.2f} ms")
print(f"P100: {np.max(arr):.2f} ms")

# 12. LLM latency fix for your current 5-second pipeline

Your current screenshot shows approximately:

`STT ≈ 2425 ms`
`FAISS ≈ 52 ms`
`Guard ≈ 1 ms`
`LLM ≈ 2594 ms`

The large LLM is therefore a major bottleneck.

For the final backend:

- Keep the model warm (`keep_alive=-1`).
- Use a small quantized model.
- Disable thinking/reasoning for short grounded QA.
- Send only 2–4 chunks to the LLM.
- Keep context around 2K tokens.
- Cap output to 32–48 tokens.
- `temperature=0`.
- Stream output for lower perceived latency.
- Benchmark on the actual deployment machine; do not claim <200 ms without measurement.

Good first Ollama candidates:
- `gemma3:1b` for minimum latency.
- `qwen3:4b` for a quality/speed compromise.

Ollama currently lists Gemma 3 1B as a small model and Gemma 3 4B at about 3.3 GB; Qwen3 4B is about 2.5 GB in Q4_K_M. citeturn319747search1turn319747search0turn319747search2

In [ ]:
llm_config = {
    "first_test_model": "gemma3:1b",
    "alternative_model": "qwen3:4b",
    "keep_alive": -1,
    "stream": True,
    "temperature": 0,
    "num_ctx": 2048,
    "num_predict": 48,
    "top_k_context": 3,
    "grounded_only": True,
}

( ROOT / "llm_latency_config.json" ).write_text(
    json.dumps(llm_config, indent=2),
    encoding="utf-8",
)
print("LLM config:", ROOT / "llm_latency_config.json")

## 13. Final artifact layout

```text
<language>/
├── faiss_ivfpq.index
├── config.json
├── checkpoint.json
└── records/
    ├── records_00000.parquet
    ├── records_00001.parquet
    └── ...
```

**FAISS** is the fast candidate-retrieval layer.

**Parquet record shards** preserve every source record and all its passages.

Do not push these large artifacts into GitHub.